# Assignment 1: Physics-Informed Neural Network for the Heat Equation (100 points)

In this assignment, you will implement a Physics-Informed Neural Network (PINN) to solve the 1D heat equation. This assignment mirrors the structure of the 2025 USAAIO Round 2 Problem 1.

## Background

The 1D heat equation describes how temperature $u(t, x)$ evolves over time:

$$u_t = \alpha u_{xx}$$

where $\alpha > 0$ is the thermal diffusivity constant.

**Domain:** $t \in [0, 1]$, $x \in [0, 1]$

**Initial condition (IC):** $u(0, x) = \sin(\pi x)$

**Boundary conditions (BC):** $u(t, 0) = 0$, $u(t, 1) = 0$ (Dirichlet)

**Thermal diffusivity:** $\alpha = 0.01$

**Analytical solution:** $u(t, x) = e^{-\alpha \pi^2 t} \sin(\pi x)$

A PINN trains a neural network $u_\theta(t, x)$ to satisfy the PDE, IC, and BC simultaneously through a compound loss function.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
import math

---

> **WARNING:** Do not modify any code outside of the designated solution areas. Do not rename or delete any provided functions or classes.

---

## Part 1: Verify the Analytical Solution (8 points)

**[Non-coding]** Prove that $u(t, x) = e^{-\alpha \pi^2 t} \sin(\pi x)$ satisfies:

1. The PDE: $u_t - \alpha u_{xx} = 0$ (4 points)
2. The initial condition: $u(0, x) = \sin(\pi x)$ (2 points)
3. The boundary conditions: $u(t, 0) = 0$ and $u(t, 1) = 0$ (2 points)

Show all intermediate steps.

### WRITE YOUR SOLUTION HERE ###



""" END OF THIS PART """

## Part 2: Build the HeatPINN Model (10 points)

**[Coding]** Implement a neural network `HeatPINN` that approximates $u(t, x)$.

**Specification:**
- Input: tensor of shape $(B, 2)$ where column 0 is $t$ and column 1 is $x$
- Architecture: Linear(2, hidden_dim) → Tanh → [Linear(hidden_dim, hidden_dim) → Tanh] × (num_layers - 1) → Linear(hidden_dim, 1)
- Output: tensor of shape $(B, 1)$
- Default: `hidden_dim=64`, `num_layers=4`

In [ ]:
### WRITE YOUR SOLUTION HERE ###

class HeatPINN(nn.Module):
    def __init__(self, hidden_dim=64, num_layers=4):
        super().__init__()
        pass  # YOUR CODE

    def forward(self, tx):
        pass  # YOUR CODE

""" END OF THIS PART """

## Part 3: Output Shape Analysis (8 points)

**[Non-coding]** For a `HeatPINN` with `hidden_dim=64` and `num_layers=4`:

1. (2 points) What is the shape of the tensor after the first linear layer (before activation)?
2. (2 points) How many total learnable parameters does the model have? Show your calculation.
3. (2 points) Why is `Tanh` used instead of `ReLU`? What property of `Tanh` is essential for PINNs?
4. (2 points) If the input has `requires_grad=True`, what happens to the gradient computation graph as data flows through the network?

### WRITE YOUR SOLUTION HERE ###



""" END OF THIS PART """

## Part 4: Create the PDE Dataset (8 points)

**[Coding]** Implement `PDEDataset` that generates random collocation points in the domain.

**Specification:**
- `n_points`: number of random $(t, x)$ points to generate
- Sample $t$ uniformly from $[0, 1]$ and $x$ uniformly from $[0, 1]$
- Store as a tensor of shape $(N, 2)$
- `__getitem__` returns a single point of shape $(2,)$

In [ ]:
### WRITE YOUR SOLUTION HERE ###

class PDEDataset(Dataset):
    def __init__(self, n_points=10000):
        pass  # YOUR CODE

    def __len__(self):
        pass  # YOUR CODE

    def __getitem__(self, idx):
        pass  # YOUR CODE

""" END OF THIS PART """

## Part 5: Create the PDE DataLoader (5 points)

**[Coding]** Create a `DataLoader` for the PDE dataset.

- Use 10,000 collocation points
- Batch size of 256
- Shuffle enabled

In [ ]:
### WRITE YOUR SOLUTION HERE ###

# Create the PDEDataset and DataLoader

""" END OF THIS PART """

## Part 6: Create the IC Dataset (8 points)

**[Coding]** Implement `ICDataset` for the initial condition.

**Specification:**
- `n_points`: number of points along $x \in [0, 1]$ at $t = 0$
- Use `torch.linspace` for $x$ values
- Store `self.tx` of shape $(N, 2)$ with $t = 0$ for all points
- Store `self.u` of shape $(N, 1)$ with $u = \sin(\pi x)$
- `__getitem__` returns `(tx[idx], u[idx])`

In [ ]:
### WRITE YOUR SOLUTION HERE ###

class ICDataset(Dataset):
    def __init__(self, n_points=100):
        pass  # YOUR CODE

    def __len__(self):
        pass  # YOUR CODE

    def __getitem__(self, idx):
        pass  # YOUR CODE

""" END OF THIS PART """

## Part 7: Create the BC Dataset (8 points)

**[Coding]** Implement `BCDataset` for the boundary conditions.

**Specification:**
- `n_points`: number of time points along $t \in [0, 1]$
- Two boundaries: $x = 0$ and $x = 1$
- Store `self.tx` of shape $(2N, 2)$ — concatenation of both boundaries
- Store `self.u` of shape $(2N, 1)$ — all zeros (Dirichlet BC)
- `__getitem__` returns `(tx[idx], u[idx])`

In [ ]:
### WRITE YOUR SOLUTION HERE ###

class BCDataset(Dataset):
    def __init__(self, n_points=100):
        pass  # YOUR CODE

    def __len__(self):
        pass  # YOUR CODE

    def __getitem__(self, idx):
        pass  # YOUR CODE

""" END OF THIS PART """

## Part 8: Configure the Optimizer (5 points)

**[Coding]** Create an instance of `HeatPINN` and configure an Adam optimizer.

- `hidden_dim=64`, `num_layers=4`
- Learning rate: $10^{-3}$
- Also create the IC and BC datasets with `n_points=100` each and extract the full tensors (`tx` and `u`) for use in training.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

# Create model, optimizer, IC data, and BC data

""" END OF THIS PART """

## Part 9: Computing Derivatives with autograd (12 points)

**[Coding]** Implement the function `compute_pde_residual` that computes the PDE residual $u_t - \alpha u_{xx}$.

**Specification:**
- Input: `model` (the HeatPINN), `tx` (tensor of shape $(B, 2)$), `alpha` (float, default 0.01)
- The function must:
  1. Enable gradients on `tx` using `.requires_grad_(True)`
  2. Compute $u = \text{model}(tx)$
  3. Use `torch.autograd.grad` with `create_graph=True` to compute $u_t$ and $u_x$
  4. Use `torch.autograd.grad` again to compute $u_{xx}$ from $u_x$
  5. Return the residual $u_t - \alpha u_{xx}$ of shape $(B, 1)$

**Important:** Use `grad_outputs=torch.ones_like(u)` and `create_graph=True`.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

def compute_pde_residual(model, tx, alpha=0.01):
    """
    Compute the PDE residual: u_t - alpha * u_xx
    
    Args:
        model: HeatPINN instance
        tx: tensor of shape (B, 2), column 0 is t, column 1 is x
        alpha: thermal diffusivity
    
    Returns:
        residual: tensor of shape (B, 1)
    """
    pass  # YOUR CODE

""" END OF THIS PART """

## Part 10: Training Loop (15 points)

**[Coding]** Implement the PINN training loop.

**Specification:**
- Train for 3000 epochs
- For each epoch, iterate over the PDE DataLoader (mini-batches)
- For each mini-batch:
  1. Compute `loss_pde` = MSE of PDE residual (should be 0)
  2. Compute `loss_ic` = MSE of model predictions at IC points vs. true IC values (use **full** IC data)
  3. Compute `loss_bc` = MSE of model predictions at BC points vs. true BC values (use **full** BC data)
  4. Total loss = `loss_pde + loss_ic + loss_bc`
  5. Backpropagate and update
- Print the total loss every 500 epochs
- Store the loss history for plotting

In [ ]:
### WRITE YOUR SOLUTION HERE ###

# Training loop
loss_history = []

""" END OF THIS PART """

## Part 11: Why Full IC/BC Data? (8 points)

**[Non-coding]** Answer the following questions:

1. (3 points) Why do we use the **full** IC and BC datasets in every training step, rather than mini-batching them like the PDE collocation points?

2. (3 points) What would happen if we mini-batched the IC data with a batch size of 10 (out of 100 IC points)? Describe the likely training behavior.

3. (2 points) The PDE collocation points are randomly sampled. Why is random sampling appropriate for the PDE loss but not for the IC/BC loss?

### WRITE YOUR SOLUTION HERE ###



""" END OF THIS PART """

## Part 12: Test and Visualize (13 points)

**[Coding]** Evaluate the trained PINN against the analytical solution.

1. (5 points) Create a 50×50 test grid over $t \in [0, 1]$ and $x \in [0, 1]$. Compute both the PINN prediction and the analytical solution on this grid.

2. (3 points) Compute and print:
   - Maximum absolute error
   - Mean absolute error
   - Relative L2 error: $\frac{\|u_{\text{pred}} - u_{\text{exact}}\|_2}{\|u_{\text{exact}}\|_2}$

3. (5 points) Create a figure with 3 subplots:
   - (a) PINN prediction as a heatmap
   - (b) Analytical solution as a heatmap
   - (c) Absolute error as a heatmap

In [ ]:
### WRITE YOUR SOLUTION HERE ###

# Evaluation and visualization

""" END OF THIS PART """